# Location Selection

In [23]:
from desdeo.problem import Constant, Variable, Problem, Objective, VariableTypeEnum, Constraint, TensorConstant, TensorVariable, ConstraintTypeEnum
import numpy as np
import pandas as pd
from slugify import slugify
import folium
# These are to just suppress warnings in the outputs of the example
import warnings

warnings.filterwarnings("ignore")

## Model inputs


In [24]:

# Mininum expected attendance to be worth visiting 
min_att = 15

# Cost constants
# The current gas costs ($/gallon)
raw_dollars_per_gallon = 3.00

# The efficiency of the vehicle (miles/gallon)
raw_mpg = 6.0
# How long the event is (hours)
hours_per_event = 4
driver_salary_per_hour = 19
raw_driver_cost_per_trip = hours_per_event * driver_salary_per_hour

# Food desert threshold (miles) (TODO this should be in minutes)
raw_food_desert_threshold = 15




## Helper functions

## Load and process Constants

In [25]:


home = "Ada"
# Read the adjacency matrix (distance in miles)
adjDist = pd.read_csv("../data/adjacencyMatrixDist.csv", index_col=0)
dist2home = adjDist.loc[:,[home]].rename(columns={home:"dist2home"})
dist2home.index.name = "city"
display("Distance to home base")
display(dist2home)

# Read the adjacency matrix (travel time in minutes)
adjTTime = pd.read_csv("../data/adjacencyMatrixTravelTime.csv", index_col=0)
display("Travel time adjacency matrix")
display(adjTTime)

# Read the cities 
cities = pd.read_csv("../data/cities.csv")
events = pd.read_csv("../data/events.csv")


'Distance to home base'

,dist2home
city,
Ada,0.00
Alger,8.81
Bluffton,19.50
Cairo,31.26
Caledonia,102.21
Carey,66.05
Columbus Grove,36.16
Continental,70.66
Cridersville,38.76


'Travel time adjacency matrix'

,Ada,Alger,Bluffton,Cairo,Caledonia,Carey,Columbus Grove,Continental,Cridersville,Delphos,...,Ottawa,Ottoville,Pandora,Prospect,Saint Marys,Spencerville,Sycamore,Upper Sandusky,Wapakoneta,Waynesfield
Ada,0.00,9.30,20.48,23.63,70.40,45.55,31.78,62.73,35.73,35.74,...,41.59,42.32,31.00,72.88,51.50,50.39,51.19,38.63,39.70,32.14
Alger,9.30,0.00,29.61,32.76,79.53,54.68,40.91,71.86,36.43,44.88,...,50.72,51.45,37.84,73.76,52.20,51.39,60.32,47.76,40.39,22.95
Bluffton,20.48,29.61,0.00,18.42,76.23,36.78,20.55,49.98,29.81,30.53,...,26.52,37.11,15.30,84.11,45.58,45.18,56.07,44.46,33.78,37.66
Cairo,23.63,32.76,18.42,0.00,78.76,51.34,11.29,42.34,26.10,17.40,...,19.81,23.98,20.73,86.64,43.96,32.74,59.54,46.98,32.15,37.08
Caledonia,70.40,79.53,76.23,78.76,0.00,47.25,87.77,118.71,95.99,91.73,...,96.46,98.31,86.98,28.59,111.76,107.07,49.45,34.24,99.96,90.16
Carey,45.55,54.68,36.78,51.34,47.25,0.00,57.96,78.84,63.76,64.48,...,55.38,71.06,48.79,53.68,79.53,79.13,19.29,15.77,67.73,71.61
Columbus Grove,31.78,40.91,20.55,11.29,87.77,57.96,0.00,32.45,35.78,27.08,...,9.92,28.86,9.43,94.52,53.64,42.42,67.42,54.87,41.84,46.76
Continental,62.73,71.86,49.98,42.34,118.71,78.84,32.45,0.00,63.28,36.13,...,23.51,26.32,41.28,126.22,74.44,52.23,97.36,86.57,69.34,74.72
Cridersville,35.73,36.43,29.81,26.10,95.99,63.76,35.78,63.28,0.00,36.77,...,44.46,43.35,35.31,101.02,28.29,30.89,74.39,61.84,16.48,27.29
Delphos,35.74,44.88,30.53,17.40,91.73,64.48,27.08,36.13,36.77,0.00,...,35.47,10.96,36.38,99.67,39.11,16.90,72.58,60.02,39.91,48.70


### Event table 

In [26]:
# Create event table
events = pd.merge(cities, events, on="city")
events.loc[:,"expectedAttendance"] = (events.loc[:,"pop"] * events.loc[:,"attendanceRate"]).astype(int)

events.loc[:, "event_id"] = events.apply(lambda row: slugify(f'{row["city"]} {row["site"]} {no_nan(row["event"])}'), axis=1)
events.loc[:, "event_pretty"] = events.apply(lambda row: f'{row["site"]} {no_nan(row["event"])}', axis=1)

# Add the distance to home for each event 
events = pd.merge(events, dist2home, on="city")

display(events)

,city,lat,long,pop,site,event,attendanceRate,dow,startTime,endTime,EventDuration,wom,Notes,expectedAttendance,event_id,event_pretty,dist2home
0,Ada,40.768056,-83.825278,5334,Public Library,NaN,0.002,*,900,1800,2.0,*,NaN,10,ada-public-library,Public Library,0.00
1,Alger,40.709722,-83.844167,837,Community Center,Food Commodities,0.002,Wednesday,900,1100,2.0,3,Wednesday after the third Tuesday each month (...,1,alger-community-center-food-commodities,Community Center Food Commodities,8.81
2,Alger,40.709722,-83.844167,837,Community Center,Community Dinner,0.002,Wednesday,1630,1830,2.0,4,Fourth Wednesday of each month,1,alger-community-center-community-dinner,Community Center Community Dinner,8.81
3,Bluffton,40.889444,-83.879167,3967,Bluffton Public Library,NaN,0.002,Thursday,900,1100,2.0,*,Bluffton Public Library,7,bluffton-bluffton-public-library,Bluffton Public Library,19.50
4,Cairo,40.830833,-84.084444,517,Public Library,NaN,0.002,*,900,1800,2.0,*,NaN,1,cairo-public-library,Public Library,31.26
5,Delphos,40.861111,-84.350000,7117,Public Library,NaN,0.002,*,900,1800,2.0,*,NaN,14,delphos-public-library,Public Library,51.03
6,Dunkirk,40.788056,-83.642778,774,HN Community Center,Community Meal,0.002,Tuesday,1630,1800,1.5,4,Fourth Wednesday of each month,1,dunkirk-hn-community-center-community-meal,HN Community Center Community Meal,17.85
7,Elida,40.786667,-84.198889,1923,Public Library,NaN,0.002,*,900,1800,2.0,*,NaN,3,elida-public-library,Public Library,44.79
8,Forest,40.805000,-83.511667,1350,Public Library,NaN,0.002,*,900,1800,2.0,*,NaN,2,forest-public-library,Public Library,35.97
9,Kenton,40.646667,-83.622500,7947,Seton Hall,NaN,0.002,Tuesday,1130,1330,2.0,4,Fourth Tuesday of each month,15,kenton-seton-hall,Seton Hall,24.99


### Close cities for events

In [27]:

close_cities = adjTTime < raw_food_desert_threshold
event2city = events.loc[:,["city", "event_id"]].merge(close_cities, left_on="city", right_index=True)
display(event2city)
event2city = (event2city.iloc[:,2:].values).astype(int)
e_adj_raw = event2city.T
display(event2city.T.shape)
e_adj_raw_list = e_adj_raw.tolist()
e_adj_raw_list

,city,event_id,Ada,Alger,Bluffton,Cairo,Caledonia,Carey,Columbus Grove,Continental,...,Ottawa,Ottoville,Pandora,Prospect,Saint Marys,Spencerville,Sycamore,Upper Sandusky,Wapakoneta,Waynesfield
0,Ada,ada-public-library,True,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,Alger,alger-community-center-food-commodities,True,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,Alger,alger-community-center-community-dinner,True,True,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,Bluffton,bluffton-bluffton-public-library,False,False,True,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,Cairo,cairo-public-library,False,False,False,True,False,False,True,False,...,False,False,False,False,False,False,False,False,False,False
5,Delphos,delphos-public-library,False,False,False,False,False,False,False,False,...,False,True,False,False,False,False,False,False,False,False
6,Dunkirk,dunkirk-hn-community-center-community-meal,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
7,Elida,elida-public-library,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
8,Forest,forest-public-library,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
9,Kenton,kenton-seton-hall,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


(36, 18)

[[1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0],
 [0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0],
 [0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [28]:
event_dist2home = events.loc[:,["dist2home"]].T.values
event_dist2home_list = event_dist2home.tolist()

## Constants

In [29]:
event_count = events.shape[0]

# Expected attenance 
raw_attendance = events.loc[:,["expectedAttendance"]] 
raw_over_attendance = (raw_attendance < min_att).astype(int)

# How many people to expect to attend each event
ea = TensorConstant(name="Expected attendance",
                                 symbol="ea", 
                                 type="int",
                                 shape=raw_attendance.T.shape,
                                 values=raw_attendance.T.values.tolist())
display(ea)

# How many events are expected to be over attended? 
eoa = TensorConstant(name="Expected over attendance", 
                    symbol="eoa", 
                    type="integer",
                    shape=raw_over_attendance.T.shape,
                    values=raw_over_attendance.T.values.tolist())

display(eoa)

# Distance to home 
ed2h = TensorConstant(name="The distance to drive from home to event", 
                    symbol="ed2h", 
                    type="real",
                    shape=event_dist2home.shape,
                    values=event_dist2home_list)
display(ed2h)

mpg = Constant(name="Miles per gallon (miles/gallon)", 
               symbol="mpg", 
               type="real",
               value=raw_mpg)

display(mpg)
dpg = Constant(name="Dollars per gallon ($/gallon)", 
               symbol="dpg", 
               type="real",
               value=raw_dollars_per_gallon)

display(dpg)
dcpt = Constant(name="Driver cost per trip ($)", 
                                symbol="dcpt", 
                                type="real", 
                                value=raw_driver_cost_per_trip)

display(dcpt)

# Event adjacency to cities
e_adj = TensorConstant(name="Event adjacency to cities", 
                      symbol="e_adj", 
                      shape=e_adj_raw.shape,
                      values=e_adj_raw_list)
display(e_adj)

city_pops = TensorConstant(name="City populations", 
                           symbol="cpop", 
                           shape=[cities.shape[0],1],
                           values=cities.loc[:,["pop"]].values.tolist()
                           )

display(city_pops)

total_pop = Constant(name="Total population of interest", 
                     symbol="tpop", 
                     type="real",
                     value=float(sum(cities.loc[:,"pop"])))

display(total_pop)


TensorConstant(name='Expected attendance', symbol='ea', shape=[1, 18], values=['List', ['List', 10, 1, 1, 7, 1, 14, 1, 3, 2, 15, 15, 15, 71, 71, 71, 71, 71, 1]])

TensorConstant(name='Expected over attendance', symbol='eoa', shape=[1, 18], values=['List', ['List', 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1]])

TensorConstant(name='The distance to drive from home to event', symbol='ed2h', shape=[1, 18], values=['List', ['List', 0.0, 8.81, 8.81, 19.5, 31.26, 51.03, 17.85, 44.79, 35.97, 24.99, 24.99, 24.99, 26.28, 26.28, 26.28, 26.28, 26.28, 39.74]])

Constant(name='Miles per gallon (miles/gallon)', symbol='mpg', value=6.0)

Constant(name='Dollars per gallon ($/gallon)', symbol='dpg', value=3.0)

Constant(name='Driver cost per trip ($)', symbol='dcpt', value=76)

TensorConstant(name='Event adjacency to cities', symbol='e_adj', shape=[36, 18], values=['List', ['List', 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], ['List', 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], ['List', 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], ['List', 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0], ['List', 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], ['List', 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], ['List', 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], ['List', 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], ['List', 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0], ['List', 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], ['List', 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0], ['List', 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0], ['List', 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0], ['List', 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], ['List'

TensorConstant(name='City populations', symbol='cpop', shape=[36, 1], values=['List', ['List', 5334], ['List', 837], ['List', 3967], ['List', 517], ['List', 560], ['List', 3565], ['List', 2160], ['List', 1102], ['List', 1791], ['List', 7117], ['List', 774], ['List', 1923], ['List', 1350], ['List', 525], ['List', 969], ['List', 1455], ['List', 7947], ['List', 676], ['List', 2177], ['List', 35579], ['List', 35999], ['List', 3046], ['List', 601], ['List', 706], ['List', 3034], ['List', 946], ['List', 4456], ['List', 966], ['List', 1204], ['List', 1067], ['List', 8397], ['List', 2198], ['List', 793], ['List', 6698], ['List', 9957], ['List', 749]])

Constant(name='Total population of interest', symbol='tpop', value=161142.0)

## Variables

In [30]:
ev = TensorVariable(
  name="Events visited",
  symbol="ev",
  variable_type=VariableTypeEnum.integer,
  shape=[events.shape[0],1],
  lowerbounds=0,
  upperbounds=1,
  initial_value=0)

display(ev)

cover = TensorVariable(
    name="Coverage of cities",
    symbol="cover", 
    variable_type=VariableTypeEnum.integer,
    shape=[adjTTime.shape[0],1],
    lowerbounds=0,
    upperbounds=1,
    initial_values=0)

cover



TensorVariable(name='Events visited', symbol='ev', variable_type=<VariableTypeEnum.integer: 'integer'>, shape=[18, 1], lowerbounds=0, upperbounds=1, initial_values=None)

TensorVariable(name='Coverage of cities', symbol='cover', variable_type=<VariableTypeEnum.integer: 'integer'>, shape=[36, 1], lowerbounds=0, upperbounds=1, initial_values=0)

## Constraints

In [31]:
# ev = [18, 1]
# cover [28, 1]
# e_adj [28, 18]
# [28, 18] . [18, 1] = [28, 1]


g = Constraint(
      name="Desert constraint",
      symbol="c",
      func="cover-e_adj@ev",
      cons_type=ConstraintTypeEnum.LTE,
      is_convex=False,
      is_linear=True,
      is_twice_differentiable=True)

## Objective

In [32]:
# Total patients seen 
total_patients = Objective(
    name = "Maximize total patients visited",
    symbol = "f_1", 
    maximize = True,
    is_twice_differentiable=True,
    func = "Sum(ea@ev)"
)

# Overstaffed events
over_staffed_events = Objective(
    name = "Minimize the number of overstaffed events",
    symbol = "f_2",
    maximize = False,
    is_twice_differentiable=True,
    func = "Sum(eoa@ev)"
)

# Cost of events
costs = Objective(
    name = "Minimize costs",
    symbol = "f_3",
    maximize = False,
    is_twice_differentiable=True,
    func = "(Sum(ed2h@ev)/mpg)*dpg + Sum(ev)*dcpt"
)


# cpop: [1, 28]
# cover: [28, 1]
# 
coverage = Objective(
    name = "Maximize coverage of region", 
    symbol = "f_4", 
    maximize = True, 
    is_twice_differentiable=True,
    func = "Sum(cover*cpop)/tpop"
)



## Problem

In [33]:
prob = Problem(
        name="Simple site selection",
        description="Simple implementation of the site selection problem",
        type="linear",
        constraints=[g],
        constants=[eoa, ea, ed2h, mpg, dpg, dcpt, e_adj, city_pops, total_pop],
        variables=[ev, cover],
        objectives=[total_patients, over_staffed_events, costs, coverage]
    )

## Ideal/nadir

In [34]:
# f_1 ideal is seeing all patients, nadir is seeing no patients
all_patients = int(np.sum(events.loc[:,"expectedAttendance"]))

# f_2 ideal is having no over staffed events
# f_2 naird is having visiting all locations with over staffed events
all_ose = int(np.sum(raw_over_attendance))


# How many miles are driven/gas costs
max_dist = events.loc[:,"dist2home"].sum()
total_gas_cost = (max_dist / raw_mpg) * raw_dollars_per_gallon
driver_cost = events.shape[0]*raw_driver_cost_per_trip
max_costs = float(driver_cost + total_gas_cost)

prob = prob.update_ideal_and_nadir(
    new_ideal={
        "f_1": all_patients,
        "f_2": 0, 
        "f_3": 0, 
        "f_4": 1.0
        }, 
    new_nadir={
        "f_1": 0,
        "f_2": all_ose,
        "f_3": max_costs, 
        "f_4": 0
        }
    )


print(f"Ideal values: {prob.get_ideal_point()}")
print(f"Nadir values: {prob.get_nadir_point()}")


Ideal values: {'f_1': 441, 'f_2': 0, 'f_3': 0, 'f_4': 1.0}
Nadir values: {'f_1': 0, 'f_2': 10, 'f_3': 1600.065, 'f_4': 0}


## RPM solver


In [35]:
from desdeo.mcdm.reference_point_method import rpm_solve_solutions
from itertools import product

ref_resolution = 2

f_1_ref = np.linspace(prob.get_ideal_point()["f_1"], prob.get_nadir_point()["f_1"],ref_resolution).tolist()
f_2_ref = np.linspace(prob.get_ideal_point()["f_2"], prob.get_nadir_point()["f_2"],ref_resolution).tolist()
f_3_ref = np.linspace(prob.get_ideal_point()["f_3"], prob.get_nadir_point()["f_3"],ref_resolution).tolist()
f_4_ref = np.linspace(prob.get_ideal_point()["f_4"], prob.get_nadir_point()["f_4"],ref_resolution).tolist()

pf_samples_raw = []
i = 0 
for ref in product(f_1_ref, f_2_ref, f_3_ref, f_4_ref):
    reference_point = {"f_1": ref[0], "f_2": ref[1], "f_3": ref[2], "f_4": ref[3]}

    print(f"Calculating for ref #{i} {reference_point}")
    try: 
        res = rpm_solve_solutions(prob, reference_point=reference_point)
        pf_samples_raw.append(res)
    except ValueError:
        print("Error running for that ref.")
    i += 1


Calculating for ref #0 {'f_1': 441.0, 'f_2': 0.0, 'f_3': 0.0, 'f_4': 1.0}
Calculating for ref #1 {'f_1': 441.0, 'f_2': 0.0, 'f_3': 0.0, 'f_4': 0.0}
Calculating for ref #2 {'f_1': 441.0, 'f_2': 0.0, 'f_3': 1600.065, 'f_4': 1.0}
Calculating for ref #3 {'f_1': 441.0, 'f_2': 0.0, 'f_3': 1600.065, 'f_4': 0.0}
Calculating for ref #4 {'f_1': 441.0, 'f_2': 10.0, 'f_3': 0.0, 'f_4': 1.0}
Calculating for ref #5 {'f_1': 441.0, 'f_2': 10.0, 'f_3': 0.0, 'f_4': 0.0}
Calculating for ref #6 {'f_1': 441.0, 'f_2': 10.0, 'f_3': 1600.065, 'f_4': 1.0}
Calculating for ref #7 {'f_1': 441.0, 'f_2': 10.0, 'f_3': 1600.065, 'f_4': 0.0}
Calculating for ref #8 {'f_1': 0.0, 'f_2': 0.0, 'f_3': 0.0, 'f_4': 1.0}
Calculating for ref #9 {'f_1': 0.0, 'f_2': 0.0, 'f_3': 0.0, 'f_4': 0.0}
Calculating for ref #10 {'f_1': 0.0, 'f_2': 0.0, 'f_3': 1600.065, 'f_4': 1.0}
Calculating for ref #11 {'f_1': 0.0, 'f_2': 0.0, 'f_3': 1600.065, 'f_4': 0.0}
Calculating for ref #12 {'f_1': 0.0, 'f_2': 10.0, 'f_3': 0.0, 'f_4': 1.0}
Calculatin

## Send relevant data to a pkl file

In [39]:
import pickle
output = open(f'../data/pf_{ref_resolution**4}.pkl', 'wb')
pickle.dump({"pf": pf_samples_raw,
             "prob" : prob,
             "cities": cities,
             "events": events,
             "event2city": event2city
             }, output)
